# Semiconductor Manufacturing Fault Detection using the SECOM Dataset

This notebook develops a classical machine learning workflow for predicting semiconductor manufacturing pass/fail outcomes from high-dimensional SECOM sensor measurements.

## Introduction

Semiconductor manufacturing is a complex process with many sensor readings (parameters) collected during production. Early detection of faulty process runs can help reduce waste, improve quality control, and support process engineers.

The SECOM dataset is useful for this task because it combines realistic industrial challenges: many features, missing values, noisy measurements, and a rare failure class.

## Problem Formulation

The task is a binary classification problem. For each process run, the model receives a vector of sensor measurements and predicts whether the run passes or fails quality control.

Let $x_i \in \mathbb{R}^p$ be the sensor feature vector for wafer $i$, where $p$ is the number of sensor measurements. Let $y_i \in \{-1, 1\}$ or equivalently $y_i \in \{0, 1\}$ be the quality label. The goal is to learn a function $f(x_i)$ that estimates the probability or class of failure while generalizing to unseen wafers.

## Dataset Description

The SECOM dataset contains anonymized sensor measurements from semiconductor manufacturing. The feature matrix includes hundreds of numeric sensor variables, while the target labels indicate normal and faulty process outcomes.

Because the variables are anonymized, the analysis focuses on statistical behavior, missingness, variance, correlations, and predictive performance rather than domain-specific sensor interpretation.

## Exploratory Data Analysis

This section will inspect the dataset shape, target distribution, basic feature summaries, and representative feature distributions. EDA will be used to identify practical preprocessing needs before model training.

Key questions include: How imbalanced is the target? Which features are mostly missing or constant? Are there obvious outliers or scaling differences across measurements?

In [ ]:
# Imports
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Define paths to the data files
DATA_DIR = Path("../data/raw")

features_path = DATA_DIR / "secom.data"
labels_path = DATA_DIR / "secom_labels.data"

# Load the data
X_raw = pd.read_csv(features_path, sep=r"\s+", header=None)
y_raw = pd.read_csv(labels_path, sep=r"\s+", header=None)

# Check the shapes of the loaded data
X_raw.shape, y_raw.shape

In [ ]:
# Display the first few rows of the features
X_raw.head()

In [ ]:
# Display the first few rows of the labels
y_raw.head()

In [ ]:
# Let's create a clean dataframe by concatenating the features and labels
feature_columns = [f"parameter_{i}" for i in range(X_raw.shape[1])]

X = X_raw.copy()
X.columns = feature_columns

labels = y_raw.copy()
labels.columns = ["target", "timestamp"]

df = X.copy()
df["target"] = labels["target"]
df["timestamp"] = labels["timestamp"]

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# Let's check for target class distribution
target_counts = df["target"].value_counts().sort_index()
target_percentages = df["target"].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})


# Create a bar plot for the target distribution
plt.figure(figsize=(6, 4))


bars = plt.bar(
    target_summary.index,
    target_summary["count"],
    width=0.35,
    color=["steelblue", "darkorange"]
)

# Add value labels on top of the bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom"
    )

# Set the title and labels for the plot
plt.title("Target Distribution: Pass vs Fail")
plt.xlabel("Target label")
plt.ylabel("Count")
plt.ylim(0, target_summary["count"].max() * 1.1)
plt.show()

The plot shows a clear class imbalance, with pass cases dominating the dataset and failure cases appearing much less frequently. This means that accuracy alone may be misleading, because a model could mostly predict the majority class and still appear to perform well.

For this reason, later modeling should use imbalance-aware evaluation metrics such as recall, precision, F1-score, confusion matrix, and precision-recall AUC. In this project, correctly identifying failed wafers is especially important, so performance on the minority class will be a key focus.

## Missing Values and Data Quality

SECOM contains many missing values, which must be handled carefully. This section will summarize missingness by row and by feature, identify columns with very high missing rates, and check for constant or near-constant measurements.

Decisions such as dropping unusable columns and imputing missing values will be based only on the training data after the train/test split.

## Preprocessing Strategy

Preprocessing will use scikit-learn Pipelines so that imputation, scaling, feature filtering, dimensionality reduction, and model fitting are applied consistently.

To avoid data leakage, every learned preprocessing step must be fit only on the training split or inside cross-validation folds. The held-out test set will be used only for final evaluation.

## Baseline Models

The first models will be classical machine learning baselines. Candidate models include logistic regression, random forest, and support vector machines.

The goal of the baseline stage is not to maximize every metric immediately, but to establish a reliable comparison point with clear preprocessing and reproducible evaluation.

## Handling Class Imbalance

Faulty wafers are expected to be much less frequent than normal wafers. Accuracy alone can therefore be misleading because a model may appear strong while missing most failures.

This section will compare strategies such as class weights, threshold adjustment, and metrics focused on the minority class.

## Model Evaluation

Model evaluation will use a held-out test set and cross-validation on the training set. Metrics will include confusion matrix, precision, recall, F1-score, ROC-AUC, and precision-recall AUC.

For this fault detection task, recall for the failure class is especially important, but it must be balanced against false alarms.

## PCA / Dimensionality Reduction

Because the dataset has hundreds of features, PCA may help reduce noise and create lower-dimensional representations for some models.

PCA will be evaluated inside a Pipeline after imputation and scaling, ensuring that the transformation is learned only from training data during cross-validation.

## Discussion and Limitations

This section will interpret model results, compare trade-offs between detecting failures and avoiding false alarms, and describe limitations of the dataset and modeling approach.

Important limitations may include anonymized features, small number of failure examples, missing values, and limited ability to infer causal process explanations.

## Conclusion

The conclusion will summarize the best-performing approach, the main lessons from the analysis, and practical recommendations for future work.

The final project should demonstrate a complete and leakage-safe classical ML workflow suitable for course review.